# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, process, and visualize data in the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) library.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nIdentifier: {metadata.identifier}\nKeywords: {getattr(metadata, 'keywords', [])}\nLicense: {getattr(metadata, 'license', '')}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id` values. This lets us see the structure of the dataset as defined in Croissant.

In [ ]:
# List all record sets and their field `@id`s
record_sets = getattr(metadata, 'record_sets', [])
if not record_sets:
    # Try the older key if present
    record_sets = getattr(metadata, 'recordSet', [])

print("Available record sets and fields:")
overview = {}
for rec_set in record_sets:
    rec_id = rec_set['@id'] if isinstance(rec_set, dict) else rec_set
    rec_obj = dataset.metadata.find_by_id(rec_id)
    print(f"RecordSet @id: {rec_id} (name: {getattr(rec_obj, 'name', '')})")
    field_ids = []
    fields = getattr(rec_obj, 'fields', []) or getattr(rec_obj, 'field', [])
    for field in fields:
        if isinstance(field, dict):
            field_ids.append(field['@id'])
        else:
            field_ids.append(field)
    print(f"  Fields: {field_ids}")
    overview[rec_id] = field_ids
    print()

if not record_sets:
    print("No record sets found in this schema. Some datasets define columns directly.")

## 3. Data Extraction
Load records from each record set into pandas DataFrames for easier analysis. We use the record set and field `@id`s discovered above.

In [ ]:
# Extract data into DataFrames by record set @id
dataframes = {}
record_set_ids = list(overview.keys())
if not record_set_ids:
    raise ValueError('No record sets discovered in schema—cannot proceed.')
for rec_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(recs)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rec_id}")
    except Exception as e:
        print(f"Could not load records for {rec_id}: {e}")

# Example: Show columns and first few records for the first available record set
main_record_set_id = record_set_ids[0]
print(f"\nColumns in record set {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll process numeric fields, filter on values, normalize, and group by categories. All fields will be referenced by their `@id`. All modifications are on pandas DataFrames extracted by `@id`.

In [ ]:
import numpy as np
from pprint import pprint

# Select a numeric field for EDA
# Let us attempt to discover a likely numeric field dynamically from the DataFrame columns
df = dataframes[main_record_set_id]
numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_field_candidates:
    # Try to coerce some likely fields
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

if not numeric_field_candidates:
    raise ValueError('No numeric fields found in main record set; cannot continue numeric EDA.')
numeric_field_id = numeric_field_candidates[0]  # Use first as an example
print(f"Using numeric field '@id': {numeric_field_id}")

# Filter records where the numeric field > threshold (e.g., 10 or pick appropriate minimum)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")
pprint(filtered_df.head().to_dict(orient='records'))

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
print(f"\nNormalized values for {numeric_field_id} (top 5):")
pprint(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head().to_dict(orient='records'))

# Group by a likely categorical/group field (by '@id')
candidate_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
# Skip group fields that are all unique
group_field_id = None
for col in candidate_group_fields:
    if df[col].nunique() < len(df) // 2:
        group_field_id = col
        break
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id} (top 5):")
    print(grouped_df.head())

## 5. Visualization
Let's plot distributions of the main numeric field and, if available, a categorical grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group field, if it exists
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load a Croissant-structured dataset with `mlcroissant` by schema URL
- Inspect metadata, available record sets, and their fields using `@id`
- Extract tabular data using `record_set` @id into pandas DataFrames
- Conduct essential EDA: filter, normalize, group, and plot distributions

This workflow can be adapted to any Croissant-compliant dataset using only the Croissant schema and the `mlcroissant` Python library.

---
For more details, see the [FAIR^2 Project](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python/latest/).